# Ingestion Smoke Test

This notebook verifies that the Neuromatch Allen Visual Behavior preprocessed parquet can be downloaded, loaded, summarized, and plotted.

## Colab setup

If running in Colab, clone or mount this repo first, then run:

```python
!pip install -r requirements-colab.txt
!pip install -e .
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from nma_data_ingestion import (
    download_neuromatch_dataset,
    load_neuromatch_dataset,
    validate_neuromatch_dataset,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [ ]:
dataset_path = download_neuromatch_dataset(data_dir=DATA_DIR, validate=True)
dataset_path

In [ ]:
data = load_neuromatch_dataset(dataset_path)
summary = validate_neuromatch_dataset(data)
summary

In [ ]:
print(f"rows: {summary.rows:,}")
print(f"columns: {summary.columns:,}")
print(f"sessions: {summary.sessions:,}")
print(f"experiments: {summary.experiments:,}")
print(f"cre lines: {summary.cre_lines}")
print(f"omitted counts: {summary.omitted_counts}")
print(f"change counts: {summary.is_change_counts}")
print(f"rewarded counts: {summary.rewarded_counts}")

In [ ]:
example_cells = data.drop_duplicates("cell_specimen_id").head(4)["cell_specimen_id"]

fig, ax = plt.subplots(figsize=(8, 4))
for cell_id in example_cells:
    row = data[data["cell_specimen_id"] == cell_id].iloc[0]
    ax.plot(row["trace_timestamps"], row["trace"], label=str(cell_id))

ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Time from stimulus or omission onset (s)")
ax.set_ylabel("dF/F")
ax.legend(title="cell_specimen_id", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()